::: {.sb-downloads}
[Download this notebook](../downloads/exercises-911-emergencies.ipynb){download="exercises-911-emergencies.ipynb"}  
Data: [911_calls_2019.csv](../downloads/data/911_calls_2019.csv){download="911_calls_2019.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

::: {.sb-lesson-links}
[← Lesson](../lessons/dbscan.ipynb)
[Solutions →](../solutions/911-emergencies.ipynb)
[All files of this section (zip)](../../files/02-dbscan.zip)
:::

Try each exercise on your own first. Stuck? Open a **hint**. Done? Compare with the **solution** at the end of each exercise.

Montgomery County, in Pennsylvania (USA), wants to know where and when its emergency services are needed most, to place ambulances and plan road safety work. It gives you a sample of the 911 calls it received in 2019. Your job: explore the calls, find the **hot spots** with DBSCAN, and make recommendations.

## The dataset 📞 {#ex-the-dataset}
`../data/911_calls_2019.csv` holds 20,000 emergency calls of 2019 (a random sample of the calls published by the county). Each row is one call:

* `time`: date and time of the call;
* `lat`, `lng`: latitude and longitude of the emergency, in degrees;
* `category`: `EMS` (medical emergency), `Fire` or `Traffic`;
* `reason`: the reason of the call, for example `CARDIAC EMERGENCY` or `VEHICLE ACCIDENT`;
* `township`: the township (town) of the call.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

import plotly.io as pio
import plotly.graph_objects as go
pio.templates["course"] = go.layout.Template(layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"])
pio.templates.default = "plotly_white+course"

EARTH_RADIUS_M = 6_371_000   # radius of the Earth, in metres
pd.set_option("display.max_colwidth", 100)

[Solution →](../solutions/911-emergencies.ipynb#ex-the-dataset){.sb-sol-link}

## Explore the calls 🔍 {#ex-explore-the-calls}

1. Load the dataset into a DataFrame called `calls`, with the `time` column read as dates. Display the first rows and the number of calls.

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `pd.read_csv(..., parse_dates=["time"])`.
:::

2. How many calls are there in each category? Show it as a bar chart.

3. Display the 10 most frequent reasons, with their category, in a horizontal bar chart. What do you notice?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `calls.groupby(["category", "reason"]).size()`, then `.nlargest(10)` and `.reset_index(name="calls")`.
:::

4. Count the calls per hour of the day and per category, and draw them as lines. Do the same per day of the week. What do you notice?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `calls["time"].dt.hour` and `calls["time"].dt.day_name()`. To keep the days in order, pass `category_orders={"day": ["Monday", ..., "Sunday"]}` to plotly.
:::

5. Draw all the calls on a map, with one colour per category. What do you see?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `px.scatter_map(calls, lat="lat", lon="lng", color="category", map_style="carto-positron", zoom=9, opacity=0.4)`.
:::

[Solution →](../solutions/911-emergencies.ipynb#ex-explore-the-calls){.sb-sol-link}

## Distances on a map 🌍 {#ex-distances-on-a-map}

6. Latitude and longitude are angles in degrees. One degree of latitude is about 111.2 km everywhere; one degree of longitude is about $111.2 \times \cos(\text{latitude})$ km. Compute how many metres 0.01° of latitude and 0.01° of longitude represent at the mean latitude of the calls. Why is a Euclidean distance on (`lat`, `lng`) a bad choice? What should you use instead?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `np.cos` needs radians: `np.cos(np.radians(latitude))`.
:::

[Solution →](../solutions/911-emergencies.ipynb#ex-distances-on-a-map){.sb-sol-link}

## Hot spots of medical emergencies 🚑 {#ex-hot-spots-of-medical-emergencies}

7. Keep only the `EMS` calls in a DataFrame called `ems`. Build `X_ems`, the array of their `[lat, lng]` coordinates **in radians**. How many EMS calls are there, and how many distinct locations?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `np.radians(ems[["lat", "lng"]].to_numpy())`; `drop_duplicates()` counts the distinct locations.
:::

8. The county defines a hot spot as **at least 20 calls in the year in a small area** (about one call every two or three weeks), so we take `min_samples=20`. Draw the k-distance plot with `k = 20`: for every call, the distance **in metres** to its 20th nearest call (itself included), sorted. Where is the elbow?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `NearestNeighbors(n_neighbors=20, metric="haversine").fit(X_ems).kneighbors(X_ems)` returns the distances (in radians) and the indices. Take the last column of the distances, multiply by `EARTH_RADIUS_M`, sort with `np.sort`. Limit the y axis to 0–3,000 m to see the elbow.
:::

9. Fit a DBSCAN with `eps` = 1,000 m (the elbow), `min_samples=20` and the haversine distance. How many clusters do you get, and what share of the calls is noise? How many calls and how many townships does the largest cluster contain? Is it a useful "hot spot"?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `DBSCAN(eps=1000 / EARTH_RADIUS_M, min_samples=20, metric="haversine", algorithm="ball_tree")`. Use `fit_predict` to get the labels, and `value_counts()` to find the largest cluster.
:::

10. The county wants hot spots of a few streets: fit DBSCAN again with `eps` = 300 m (and still `min_samples=20`). Store the labels in a new column `cluster` of `ems`. How many hot spots do you find, and what share of the calls is noise?

11. Draw the calls that belong to a hot spot on a map (leave the noise out), with one colour per cluster.

::: {.callout-tip collapse="true" title="Show hint"}
Hint: filter with `ems[ems["cluster"] != -1]`; convert the cluster number to text (`.astype(str)`) so that plotly uses distinct colours instead of a colour scale.
:::

12. Describe the hot spots: for each cluster, compute the number of calls, the centre (mean latitude and longitude), the main township, the number of townships and the 3 most frequent reasons. Display the 8 largest. Then draw one bubble per hot spot on a map, sized by the number of calls. What do you notice?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: `ems[ems["cluster"] != -1].groupby("cluster").agg(calls=("reason", "size"), lat=("lat", "mean"), ...)`. For the main township and the top reasons, pass a function, for example `lambda r: ", ".join(r.value_counts().index[:3])`.
:::

[Solution →](../solutions/911-emergencies.ipynb#ex-hot-spots-of-medical-emergencies){.sb-sol-link}

## Hot spots of traffic incidents 🚗 {#ex-hot-spots-of-traffic-incidents}

13. Do the same for the `Traffic` calls: keep them in `traffic`, fit DBSCAN with the same definition of a hot spot (300 m, `min_samples=20`), count the hot spots and the noise, map the calls in a hot spot, and describe the 8 largest hot spots. How do they differ from the EMS hot spots?

::: {.callout-tip collapse="true" title="Show hint"}
Hint: reuse the code of questions 7, 10, 11 and 12; the function `describe_hot_spots` works on any DataFrame with a `cluster` column.
:::

[Solution →](../solutions/911-emergencies.ipynb#ex-hot-spots-of-traffic-incidents){.sb-sol-link}

## Recommendations 📋 {#ex-recommendations}

14. Using everything you found, what would you recommend to the county? Write 3 or 4 recommendations.

To go further: find the locations of the ambulance and fire stations of the county (they are public) and compare them with the hot spots. Are some hot spots far from any station?

[Solution →](../solutions/911-emergencies.ipynb#ex-recommendations){.sb-sol-link}